# Chosen-value RSA — group results

Conditions relabel each trial by which stimulus was actually **chosen** (`chosen_stim`),
not which was cued first — see `run_subject_chosen()` in `run_rsa_roi.py`
(`--condition-on chosen`). Motivation: GLMsingle cue-locked betas are whole-trial
contaminated (session-notes 2026-09-03 findings 18-19), so this asks whether they encode
the value of the eventually-**chosen** item — the standard univariate "chosen value"
signal — rather than just the cued image's value.

Two trees, both n=58 (`participants_mvpa.tsv` minus `sub-46`, absent from the BBT):

- **`rsa_chosen_stim2`** (`--chosen-scope stim2`) — restricted to trials where the chosen
  item was the *second* stimulus. The cleanest test: the beta is locked to a *different*
  image's onset entirely, so any structure found can't be repackaged cue/visual-value
  coding. Pooled-across-runs scope only (per-run stim2-only counts too sparse — checked
  offline pre-submission).
- **`rsa_chosen_all`** (`--chosen-scope all`) — pools both roles with a `role_fraction`
  nuisance regressor, plus per-run (`learning1`/`learning2`/`test`) scopes where a
  subject's counts support it — lets learning-vs-test be compared directly, same as the
  stim1-conditioned pipeline.

Condition universe fixed to the **nonfigure stimuli** (values {2,3,4}, 6 identities) —
not dynamic per-subject dropping — reusing the design's existing figure-confound boundary
(session-notes 2026-08-26 finding 2). Chosen-conditions are not presentation-balanced
across value the way stim1-conditions are (high-value stimuli get chosen far more often);
crossnobis is unbiased regardless of N per condition, so this adds sampling noise, not
systematic bias, but the all-8-stimuli universe left 33/62 dev_sample subjects short of
the CV-fold floor vs. 0/62 for nonfigure-only, hence the restriction.

Model terms: `category`, `value` (**the chosen-value RDM**), `frequency` (identity-level
choice-frequency design label, same as stim1-mode), `unchosen_value` (partner's value,
confound control), `role_fraction` (fraction of a condition's trials where the chosen item
was also the cue — 0 by construction under `chosen_scope='stim2'`, auto-dropped there).
Only the `objective` value/frequency variant is computed (no rl/ck graded analogs yet —
`chosen_value_rl`/`chosen_value_ck` already exist in the BBT for that extension).

See `session-notes/2026-09-07_chosen-value-rsa-and-glm-definition-check.md` for the full
session log.


## ⚠️ Two caveats to read before the numbers below

**1. The frequency effect below is NOT habit/learning evidence.** Same status as every
other RSA notebook in this repo — see
`session-notes/2026-09-07_frequency-confound-sweep-and-verdict.md`. β(frequency) is
present at full strength in the first half of `learning1`, before differential
reinforcement could plausibly have accrued, and an extensive confound sweep found nothing
that explains its early presence.

**2. "Chosen value" and "cued value" are not cleanly separable in this beta.** At
TR=2.33s, first-stim, second-stim (~0.8s later) and the response (~1.7s later) fall in
the *same TR* under floor-division onset assignment (a documented same-TR collision —
`dev_glmsingle_stim_cat.ipynb` Step 4), so only 8 first-stimulus conditions are modeled
and `stimdur` is deliberately set to span first-stim-onset-to-response (confirmed 1.42s
mean) specifically to cover both stimulus presentations. This is the confirmed,
by-design mechanism behind the whole-trial contamination this whole analysis is built on
(session-notes 2026-09-03 findings 18-19) — but it also means a single trial's beta can't
in principle cleanly separate "responded to the cued image" from "value of whichever item
won the choice." The `chosen_scope='stim2'` restriction is the best available control for
this (the beta is locked to a *different* image's onset than the chosen one), not a full
solution. See `session-notes/2026-09-07_chosen-value-rsa-and-glm-definition-check.md`
finding 4-5 for the full derivation, including a test that rules out one specific
consequence (fixed-stimdur duration mismatch) as an explanation for the frequency effect.


In [1]:
import glob
import pandas as pd
import numpy as np
from scipy import stats

DERIV = '/Users/hugofluhr/phd_local/data/LearningHabits/derivatives'
MASK_ORDER = ['wholebrain', 'visualcortex', 'fusiform', 'vmpfc', 'striatum',
              'habit', 'putamen', 'premotor', 'parietal']

def load(tree):
    files = glob.glob(f'{DERIV}/{tree}/sub-*/sub-*_rsa_chosen_results.csv')
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df['mask'] = pd.Categorical(df['mask'], categories=MASK_ORDER, ordered=True)
    return df.sort_values(['mask', 'scope']).reset_index(drop=True)

stim2 = load('rsa_chosen_stim2')
allr  = load('rsa_chosen_all')
print(f"stim2 tree: {stim2.shape[0]} rows, {stim2['subject'].nunique()} subjects, "
      f"scopes={list(stim2['scope'].unique())}")
print(f"all   tree: {allr.shape[0]} rows, {allr['subject'].nunique()} subjects, "
      f"scopes={list(allr['scope'].unique())}")


stim2 tree: 522 rows, 58 subjects, scopes=['pooled_chosen']
all   tree: 1944 rows, 58 subjects, scopes=['learning1', 'learning2', 'pooled_chosen', 'test']


In [2]:
TERMS = ['category', 'value', 'frequency', 'unchosen_value', 'role_fraction']

def group_stats(df, terms=TERMS):
    """One-sample t-test of each beta_<term> against 0, per (mask, scope).

    Betas are standardised partial regression weights (RDM and predictors z-scored per
    subject) — comparable across subjects/ROIs, and testable against 0 because
    crossnobis is unbiased around 0 under the null (crossnobis_validation.ipynb).
    A term that's constant for every subject in a (mask, scope) cell (e.g.
    role_fraction under chosen_scope='stim2', which is 0 by construction) is skipped.
    """
    rows = []
    for (mask, scope), g in df.groupby(['mask', 'scope'], observed=True):
        row = {'mask': mask, 'scope': scope, 'n': len(g)}
        for t in terms:
            col = f'beta_{t}'
            if col not in g or g[col].isna().all():
                continue
            x = g[col].dropna().values
            if len(x) < 3 or np.ptp(x) == 0:
                continue
            tstat, p = stats.ttest_1samp(x, 0)
            row[f'{t}_mean'], row[f'{t}_t'], row[f'{t}_p'] = x.mean(), tstat, p
        rows.append(row)
    return pd.DataFrame(rows)

stim2_stats = group_stats(stim2)
all_stats   = group_stats(allr)


## Stage 1 — `chosen_scope='stim2'`, the cleanest test

Only trials where the chosen item was the *second* stimulus. `pooled_chosen` is the only
scope (per-run counts too sparse for this restriction).


In [3]:
cols = ['mask', 'scope', 'n', 'value_mean', 'value_t', 'value_p',
        'frequency_mean', 'frequency_t', 'frequency_p',
        'category_mean', 'category_t', 'category_p']
stim2_stats[cols].sort_values('value_p')


,mask,scope,n,value_mean,value_t,value_p,frequency_mean,frequency_t,frequency_p,category_mean,category_t,category_p
4,striatum,pooled_chosen,58,0.097436,1.111326,0.271094,0.019187,0.608718,0.545129,-0.000132,-0.004525,0.996406
6,putamen,pooled_chosen,58,-0.084740,-1.002194,0.320485,-0.024348,-0.729876,0.468455,-0.047986,-1.630834,0.108439
5,habit,pooled_chosen,58,-0.081365,-0.933537,0.354480,-0.041483,-1.246844,0.217554,-0.048512,-1.519462,0.134174
0,wholebrain,pooled_chosen,58,-0.077928,-0.924104,0.359328,-0.016882,-0.612151,0.542872,-0.014900,-0.430624,0.668366
1,visualcortex,pooled_chosen,58,-0.066715,-0.745172,0.459229,0.064774,2.190469,0.032595,-0.029433,-0.998689,0.322166
7,premotor,pooled_chosen,58,-0.050525,-0.702449,0.485258,-0.010015,-0.328989,0.743369,0.000543,0.017490,0.986107
3,vmpfc,pooled_chosen,58,0.018732,0.229357,0.819413,-0.037223,-1.270656,0.209013,0.027545,0.961008,0.340607
2,fusiform,pooled_chosen,58,-0.008386,-0.089231,0.929211,0.192554,5.418268,0.000001,-0.040003,-1.344186,0.184214
8,parietal,pooled_chosen,58,-0.005136,-0.067783,0.946195,0.010319,0.355566,0.723478,0.050990,1.578958,0.119880


**Chosen-value: null.** Best case is striatum (t=1.11, p=.27) — not close to significant,
and nothing survives correction across the 9 masks even before considering that. Directions
are mixed, not systematically positive. **Frequency: fusiform t=5.42, p=1e-6**, visual
cortex t=2.19, p=.033 — the same effect as the stim1-conditioned pipeline, replicating
under a completely different (chosen-identity) condition definition and the most
conservative scope available. `category` shows nothing here.


## Stage 2 — `chosen_scope='all'`, pooled + per-run

Both roles pooled (with `role_fraction` as a nuisance regressor), plus per-run scopes
where a subject's counts supported it (52-60/58 subjects per run — see
`run_subject_chosen()` docstring). This is what lets learning-vs-test be compared
directly.


In [4]:
print("=== value (the chosen-value RDM) ===")
display(all_stats[cols].sort_values('value_p'))


=== value (the chosen-value RDM) ===


,mask,scope,n,value_mean,value_t,value_p,frequency_mean,frequency_t,frequency_p,category_mean,category_t,category_p
9,fusiform,learning2,56,-0.313373,-2.565681,0.013053,0.284964,8.367243,2.206977e-11,-0.031685,-1.020887,0.311778
5,visualcortex,learning2,56,-0.300084,-2.128935,0.037753,0.169532,5.254416,2.485639e-06,0.029936,0.865307,0.390629
6,visualcortex,pooled_chosen,58,-0.217741,-1.949761,0.056130,0.181779,5.680107,4.776528e-07,0.003511,0.097908,0.922349
2,wholebrain,pooled_chosen,58,-0.174828,-1.690704,0.096355,0.017866,0.664885,5.088050e-01,-0.017760,-0.541917,0.589988
34,parietal,pooled_chosen,58,-0.185191,-1.665663,0.101268,-0.044869,-1.349053,1.826548e-01,0.013899,0.394095,0.694981
30,premotor,pooled_chosen,58,-0.155860,-1.504480,0.137979,-0.085807,-2.377689,2.079842e-02,-0.009406,-0.286645,0.775424
26,putamen,pooled_chosen,58,0.136461,1.411303,0.163590,-0.009345,-0.282960,7.782326e-01,-0.030978,-0.922788,0.360008
21,habit,learning2,56,-0.218195,-1.235219,0.222000,-0.024468,-0.816278,4.178623e-01,0.006190,0.174511,0.862105
31,premotor,test,53,-0.083207,-1.155996,0.252967,0.003844,0.144504,8.856610e-01,-0.010614,-0.324974,0.746505
24,putamen,learning1,49,-0.108116,-1.110187,0.272449,0.005870,0.118726,9.059878e-01,0.003702,0.098026,0.922320


Still null: best nominal hits are fusiform/`learning2` (t=-2.57, p=.013) and
visualcortex/`learning2` (p=.038), both **negative**-signed (more value-different pairs →
*more* similar patterns) — the wrong direction for a value-coding interpretation, and
neither replicates across scopes or masks. No vmPFC/striatum signal, which is where the
univariate chosen-value literature would predict it. Treating this as a genuine null, not
an underpowered one (n=58, standardised partial betas, same pipeline that detects the
frequency effect at p<1e-13 below).


In [5]:
fcols = ['mask', 'scope', 'n', 'frequency_mean', 'frequency_t', 'frequency_p']
print("=== frequency, by scope (learning1 -> learning2 -> test dissociation) ===")
for m in ['fusiform', 'visualcortex', 'wholebrain']:
    sub = all_stats[all_stats['mask'] == m][fcols].set_index('scope')
    sub = sub.reindex(['learning1', 'learning2', 'pooled_chosen', 'test'])
    print(f"\n{m}:")
    display(sub)


=== frequency, by scope (learning1 -> learning2 -> test dissociation) ===

fusiform:


,mask,n,frequency_mean,frequency_t,frequency_p
scope,,,,,
learning1,fusiform,49,0.353310,7.550190,1.060179e-09
learning2,fusiform,56,0.284964,8.367243,2.206977e-11
pooled_chosen,fusiform,58,0.286530,9.829464,7.036414e-14
test,fusiform,53,0.045182,1.470270,1.475140e-01



visualcortex:


,mask,n,frequency_mean,frequency_t,frequency_p
scope,,,,,
learning1,visualcortex,49,0.230984,6.404039,6.035795e-08
learning2,visualcortex,56,0.169532,5.254416,2.485639e-06
pooled_chosen,visualcortex,58,0.181779,5.680107,4.776528e-07
test,visualcortex,53,0.038854,1.302686,1.984216e-01



wholebrain:


,mask,n,frequency_mean,frequency_t,frequency_p
scope,,,,,
learning1,wholebrain,49,0.111475,2.747562,0.008432
learning2,wholebrain,56,0.051961,2.085190,0.041706
pooled_chosen,wholebrain,58,0.017866,0.664885,0.508805
test,wholebrain,53,0.011735,0.410270,0.683293


**The learning→test dissociation replicates too.** Fusiform: p≤1e-9 in `learning1`/
`learning2`/`pooled_chosen`, drops to p=.15 in `test`. Visual cortex: p≤3e-6 in
`learning1`/`learning2`/`pooled_chosen`, drops to p=.20 in `test`. This is independent
evidence — a completely different condition definition (chosen identity, not cue
identity) — that both the frequency effect itself and its vanishing in `test`
(companion note open thread 2, still unresolved) are not artifacts specific to the
stim1-conditioned pipeline.


In [6]:
ncols = ['mask', 'scope', 'n', 'unchosen_value_t', 'unchosen_value_p',
         'role_fraction_t', 'role_fraction_p']
print("=== nuisance terms, pooled_chosen scope ===")
n1 = stim2_stats[stim2_stats['scope'] == 'pooled_chosen'][
    ['mask', 'scope', 'n', 'unchosen_value_t', 'unchosen_value_p']]
n2 = all_stats[all_stats['scope'] == 'pooled_chosen'][ncols]
print("\nstim2 tree (role_fraction constant=0, auto-dropped):")
display(n1.sort_values('unchosen_value_p'))
print("\nall tree:")
display(n2.sort_values('unchosen_value_p'))


=== nuisance terms, pooled_chosen scope ===

stim2 tree (role_fraction constant=0, auto-dropped):


,mask,scope,n,unchosen_value_t,unchosen_value_p
1,visualcortex,pooled_chosen,58,2.131429,0.037378
0,wholebrain,pooled_chosen,58,0.913931,0.364603
4,striatum,pooled_chosen,58,-0.875817,0.384805
2,fusiform,pooled_chosen,58,0.587909,0.558915
7,premotor,pooled_chosen,58,0.559334,0.578125
3,vmpfc,pooled_chosen,58,-0.508343,0.613175
8,parietal,pooled_chosen,58,0.351046,0.726848
6,putamen,pooled_chosen,58,0.254404,0.800099
5,habit,pooled_chosen,58,0.019112,0.984818



all tree:


,mask,scope,n,unchosen_value_t,unchosen_value_p,role_fraction_t,role_fraction_p
6,visualcortex,pooled_chosen,58,3.107271,0.002943,0.859441,0.393697
2,wholebrain,pooled_chosen,58,1.947888,0.056359,-0.166862,0.868069
34,parietal,pooled_chosen,58,1.910171,0.061147,1.046412,0.299788
30,premotor,pooled_chosen,58,1.650450,0.104351,0.799520,0.427308
26,putamen,pooled_chosen,58,-1.397268,0.167749,-1.031741,0.306551
10,fusiform,pooled_chosen,58,1.206656,0.232549,1.063513,0.292034
14,vmpfc,pooled_chosen,58,0.733358,0.466346,-0.468576,0.641160
18,striatum,pooled_chosen,58,-0.284543,0.777025,0.600669,0.550441
22,habit,pooled_chosen,58,0.073136,0.941954,-0.112520,0.910806


`role_fraction` is never significant (not soaking up spurious pooled-role structure).
`unchosen_value` (the partner's value) shows a mild visual-cortex effect (p=.003 `all`
tree, p=.037 `stim2` tree) — plausible low-level partner-value leakage into visual
regions, not a follow-up priority unless it recurs.

---

## Findings summary

1. **No chosen-value RSA signal anywhere** — 9 masks × up to 4 scopes × 2 chosen-scope
   trees, no `beta_value` survives correction, best case p=.27 (striatum, cleanest
   `stim2`-only test). Directions are mixed/negative where nominal, not a coding pattern.
   A genuine null, not underpowered — same subjects/pipeline detects the frequency effect
   at p as low as 7e-14.
2. **The unexplained β(frequency) effect replicates under an orthogonal condition
   definition** — chosen identity instead of cue identity — including its **learning→test
   attenuation** (strong p≤1e-9 in learning1/learning2, ns in test, fusiform and visual
   cortex both). Independent evidence the effect (and its most puzzling property, the
   vanishing in `test`) is not an artifact of the stim1-conditioned pipeline specifically.
   Still not explained — see companion note open thread 2.
3. **GLM stimdur duration-mismatch tested and ruled out** as an explanation for the
   frequency effect specifically (RT vs. choice-frequency label r≈0, ns in every run) —
   see `session-notes/2026-09-07_chosen-value-rsa-and-glm-definition-check.md` finding 5.
   The general TR-collision/stimdur-span fact (finding 4, same note) remains a real
   interpretive ceiling on what this notebook's "chosen value" can mean, independent of
   that specific mechanism being ruled out.
4. **Nuisance terms behave as expected** — `role_fraction` never significant,
   `unchosen_value` shows only a mild visual-cortex leak.

**Status:** run + analyzed 2026-09-07, n=58 both trees. rl/ck graded chosen-value variants
not yet computed (see session note open thread 2).
